# GGUF / llama.cpp Inference

This notebook loads a local GGUF model with `llama-cpp-python`, benchmarks tokens per second, and compares it to a small HF fallback path.

In [ ]:
from pathlib import Path
import os
import sys
import time

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'llm_lab').exists():
        sys.path.insert(0, str(candidate))
        repo_root = candidate
        break

from llm_lab.benchmarks import measure_generation
from llm_lab.models import load_causal_lm

prompt = 'Explain RAG in 3 lines.'
gguf_path = os.environ.get('GGUF_MODEL_PATH', '')
print(f'Repo root: {repo_root}')
print(f"GGUF path: {gguf_path or 'not set'}")

In [ ]:
hf_bundle = load_causal_lm(os.environ.get('LLM_LAB_GGUF_COMPARE_MODEL', 'sshleifer/tiny-gpt2'), quantized=False, device_map=None)
hf_result = measure_generation(hf_bundle.model, hf_bundle.tokenizer, prompt)
hf_result

In [ ]:
llama_cpp_result = {'status': 'not run'}
if gguf_path:
    try:
        from llama_cpp import Llama

        llm = Llama(model_path=gguf_path, n_ctx=2048, n_threads=max(os.cpu_count() or 2, 2))
        start = time.perf_counter()
        completion = llm(prompt, max_tokens=100, temperature=0.2, top_p=0.95)
        elapsed = time.perf_counter() - start
        text = completion['choices'][0]['text']
        tokens_per_second = 100 / max(elapsed, 1e-9)
        llama_cpp_result = {'elapsed_seconds': elapsed, 'tokens_per_second': tokens_per_second, 'text': text}
    except Exception as exc:
        llama_cpp_result = {'status': 'failed', 'reason': str(exc)}
llama_cpp_result

The `llm_lab.gguf_api` module exposes the same model load path as a FastAPI service with `/health` and `/generate` endpoints. Point `GGUF_MODEL_PATH` at a local `.gguf` file and run `uvicorn llm_lab.gguf_api:app --reload`.